In [ ]:
import os
import random
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset,TensorDataset,SubsetRandomSampler
from sklearn.metrics import classification_report
from torchvision import transforms
from pywt import dwt2, idwt2, wavedec2
import torchvision.models as models
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support
import seaborn as sns
from sklearn.metrics import precision_recall_curve,f1_score,roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_curve,f1_score,roc_curve,roc_auc_score,auc,accuracy_score,average_precision_score,precision_score,recall_score
from sklearn.preprocessing import label_binarize
import warnings
from sklearn.preprocessing import MinMaxScaler, StandardScaler
warnings.filterwarnings("ignore")
import torchvision
from torchvision import models
import torch.nn.functional as F
import torch.nn.init as init

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def make_img(t_img):
    img = pd.read_pickle(t_img)
    img_l = []
    for i in range(len(img)):
        img_l.append(img.values[i][0])
    
    return np.array(img_l)

In [ ]:
def weight_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.LayerNorm):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

In [ ]:
# # 设置随机种子
# def reset_random_seeds(seed):
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        # torch.cuda.manual_seed_all(seed)  # 如果使用多个GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def plot_classification_report(y_tru, y_prd, mode, learning_rate, batch_size,epochs, figsize=(7, 7), ax=None):

    plt.figure(figsize=figsize)

    xticks = ['precision', 'recall', 'f1-score', 'support']
    yticks = ["Control", "Moderate", "Alzheimer's" ] 
    yticks += ['avg']

    rep = np.array(precision_recall_fscore_support(y_tru, y_prd)).T
    avg = np.mean(rep, axis=0)
    avg[-1] = np.sum(rep[:, -1])
    rep = np.insert(rep, rep.shape[0], avg, axis=0)

    sns.heatmap(rep,
                annot=True, 
                cbar=False, 
                xticklabels=xticks, 
                yticklabels=yticks,
                ax=ax, cmap = "Blues")
    
    plt.savefig('report_' + str(mode) + '_' + str(learning_rate) +'_' + str(batch_size)+'_' + str(epochs)+'.png')

In [ ]:
def calc_confusion_matrix(result, test_label,mode, learning_rate, batch_size, epochs):
    result = F.one_hot(result,num_classes=4)
    # print(result)

    test_label = F.one_hot(test_label,num_classes=4)
    # print(test_label)

    true_label= np.argmax(test_label, axis =1)

    predicted_label= np.argmax(result, axis =1)
    
    n_classes = 4
    precision = dict()
    recall = dict()
    thres = dict()
    for i in range(n_classes):
        precision[i], recall[i], thres[i] = precision_recall_curve(test_label[:, i],
                                                            result[:, i])


    print ("Classification Report :") 
    print (classification_report(true_label, predicted_label))
    cr = classification_report(true_label, predicted_label, output_dict=True)
    return cr, precision, recall, thres

In [ ]:
class create_modal_plasma(nn.Module):
    def __init__(self,transcriptomicsinput,num_classes=50):
        super(create_modal_plasma, self).__init__()
        self.fc1 = nn.Linear(transcriptomicsinput, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(256)
        self.batch_norm2 = nn.BatchNorm1d(128)
        self.batch_norm3 = nn.BatchNorm1d(64)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x))
        x = self.batch_norm1(x)
        x = self.dropout1(x)
        x = F.leaky_relu(self.fc2(x))
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        x = F.leaky_relu(self.fc3(x))
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

In [ ]:
# 通道注意力机制
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

# 空间注意力机制
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

# 通道注意力 + 空间注意力机制
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x
    
class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# 多尺度卷积块
class MultiScaleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MultiScaleConv, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2)
        self.conv5 = nn.Conv2d(in_channels, out_channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm2d(out_channels * 3)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv3(x)
        x3 = self.conv5(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.bn(x)
        return x
    
# 结合注意力机制的CNN模型
class AttentionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(AttentionCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.cbam1 = CBAM(64)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.cbam2 = CBAM(128)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.cbam3 = CBAM(256)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(256 * 9 * 9, num_classes)  # 假设输入图像大小为 (3, 32, 32)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.cbam1(x)
        x = self.maxpool(x)

        x = self.relu(self.conv2(x))
        x = self.cbam2(x)
        x = self.maxpool(x)

        x = self.relu(self.conv3(x))
        x = self.cbam3(x)
        x = self.maxpool(x)
        # print(x.shape)
        x = self.flatten(x)
        x = self.fc(x)
        return x
    
class create_modal_img(nn.Module):
    def __init__(self, num_classes=10):
        super(create_modal_img, self).__init__()
        self.multi_scale_conv1 = MultiScaleConv(3, 64)
        self.se1 = SELayer(64 * 3)

        self.multi_scale_conv2 = MultiScaleConv(64 * 3, 100)
        self.se2 = SELayer(100 * 3)

        self.multi_scale_conv3 = MultiScaleConv(100 * 3, 50)
        self.se3 = SELayer(50 * 3)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(50 * 3 * 9 * 9, num_classes)  # 假设输入图像大小为 (3, 32, 32)

    def forward(self, x):
        x = self.relu(self.multi_scale_conv1(x))
        x = self.se1(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv2(x))
        x = self.se2(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv3(x))
        x = self.se3(x)
        x = self.maxpool(x)
        # print(x.shape)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x
    
class new_AttentionMultiScaleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(new_AttentionMultiScaleCNN, self).__init__()
        self.multi_scale_conv1 = MultiScaleConv(3, 64)
        self.se1 = SELayer(64 * 3)

        self.multi_scale_conv2 = MultiScaleConv(64 * 3, 100)
        self.se2 = SELayer(100 * 3)

        self.multi_scale_conv3 = MultiScaleConv(100 * 3, 50)
        self.se3 = SELayer(50 * 3)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(50 * 3 * 9 * 9, num_classes)  # 假设输入图像大小为 (3, 32, 32)

    def forward(self, x):
        x = self.relu(self.multi_scale_conv1(x))
        x = self.se1(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv2(x))
        x = self.se2(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv3(x))
        x = self.se3(x)
        x = self.maxpool(x)
        # print(x.shape)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=100, kernel_size=3, stride=1)
        self.relu = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout1 = nn.Dropout(p=0.5)
        self.conv2 = nn.Conv2d(in_channels=100, out_channels=50, kernel_size=3, stride=1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout2 = nn.Dropout(p=0.3)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(50 * 16 * 16, 4)  # Assuming input shape (72, 72, 1) after convolutions and pooling

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool1(x)
        x = self.dropout1(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool2(x)
        x = self.dropout2(x)
        x = self.flatten(x)
        x = self.fc(x)
        out=torch.softmax(x, dim=1)
        return out

In [ ]:
# class NeuralNetwork(nn.Module):
#     def __init__(self,input,num_classes=10):
#         super(NeuralNetwork, self).__init__()
#         self.fc0 = nn.Linear(input, 512)
#         self.fc1 = nn.Linear(512, 256)
#         self.fc2 = nn.Linear(256, 128)
#         self.fc3 = nn.Linear(128, 64)
#         self.fc4 = nn.Linear(64, num_classes)
#         self.dropout0 = nn.Dropout(0.5)
#         self.dropout1 = nn.Dropout(0.5)
#         self.dropout2 = nn.Dropout(0.3)
#         self.dropout3 = nn.Dropout(0.2)
#         self.batch_norm0 = nn.BatchNorm1d(512)
#         self.batch_norm1 = nn.BatchNorm1d(256)
#         self.batch_norm2 = nn.BatchNorm1d(128)
#         self.batch_norm3 = nn.BatchNorm1d(64)

#     def forward(self, x):
#         x = F.leaky_relu(self.fc0(x))
#         x = self.batch_norm0(x)
#         x = self.dropout0(x)
#         x = F.leaky_relu(self.fc1(x))
#         x = self.batch_norm1(x)
#         x = self.dropout1(x)
#         x = F.leaky_relu(self.fc2(x))
#         x = self.batch_norm2(x)
#         x = self.dropout2(x)
#         x = F.leaky_relu(self.fc3(x))
#         x = self.batch_norm3(x)
#         x = self.dropout3(x)
#         x = self.fc4(x)
#         return F.softmax(x, dim=1)
class NeuralNetwork(nn.Module):
    def __init__(self,input,num_classes=6):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(input, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 6)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(256)
        self.batch_norm2 = nn.BatchNorm1d(128)
        self.batch_norm3 = nn.BatchNorm1d(64)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x))
        x = self.batch_norm1(x)
        x = self.dropout1(x)
        x = F.leaky_relu(self.fc2(x))
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        x = F.leaky_relu(self.fc3(x))
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return F.softmax(x, dim=1)

In [ ]:
class FC(nn.Module):
    def __init__(self, in_size, out_size, dropout_r=0., use_relu=True):
        super(FC, self).__init__()
        self.dropout_r = dropout_r
        self.use_relu = use_relu

        self.linear = nn.Linear(in_size, out_size)

        if use_relu:
            self.relu = nn.ReLU(inplace=True)

        if dropout_r > 0:
            self.dropout = nn.Dropout(dropout_r)

    def forward(self, x):
        x = self.linear(x)

        if self.use_relu:
            x = self.relu(x)

        if self.dropout_r > 0:
            x = self.dropout(x)

        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_size, mid_size, out_size, dropout_r=0., use_relu=True):
        super(MLP, self).__init__()

        self.fc = FC(in_size, mid_size, dropout_r=dropout_r, use_relu=use_relu)
        self.linear = nn.Linear(mid_size, out_size)

    def forward(self, x):
        return self.linear(self.fc(x))

In [ ]:
class FFN(nn.Module):
    def __init__(self,):
        super(FFN, self).__init__()

        self.mlp = MLP(
            in_size=50,
            mid_size=100,
            out_size=50,
            dropout_r=0.3,
            use_relu=True
        )

    def forward(self, x):
        return self.mlp(x)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self):
        super(SelfAttention, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=50, num_heads=2)
        self.dropout1 = nn.Dropout(0.3)
        self.layer_norm1 = nn.LayerNorm(50)
        self.dropout2 = nn.Dropout(0.2)
        self.layer_norm2 = nn.LayerNorm(50)
        self.ffn=FFN()

    def forward(self, inputs):
        inputs=torch.unsqueeze(inputs, dim=1)
        outputs, _ = self.multihead_attn(inputs, inputs, inputs, attn_mask=None)
        outputs = self.dropout1(outputs)
        outputs = self.layer_norm1(inputs + outputs)
        outputs_2=self.ffn(outputs)
        outputs_2 = self.dropout2(outputs_2)
        all_outputs=self.layer_norm2(outputs + outputs_2)
        return all_outputs[:,0,:]
        # return outputs

In [ ]:
class FeatureWeightedAdd(nn.Module):
    def __init__(self, input_dim1, input_dim2, output_dim):
        super(FeatureWeightedAdd, self).__init__()
        self.fc1 = nn.Linear(input_dim1, output_dim)
        self.fc2 = nn.Linear(input_dim2, output_dim)
        self.weight1 = nn.Parameter(torch.tensor(0.5))
        self.weight2 = nn.Parameter(torch.tensor(0.5))

    def forward(self, x1, x2):
        fused = self.weight1 * self.fc1(x1) + self.weight2 * self.fc2(x2)
        return fused
class FeatureWeightedFusion(nn.Module):
    def __init__(self, input_dim1, input_dim2, output_dim):
        super(FeatureWeightedFusion, self).__init__()
        self.fc1 = nn.Linear(input_dim1, output_dim)
        self.fc2 = nn.Linear(input_dim2, output_dim)
        self.weight1 = nn.Parameter(torch.tensor(0.5))
        self.weight2 = nn.Parameter(torch.tensor(0.5))

    def forward(self, x1, x2):
        # fused = self.weight1 * self.fc1(x1) + self.weight2 * self.fc2(x2)
        return self.weight1 * self.fc1(x1) , self.weight2 * self.fc2(x2)
class FeatureWeightedFusion2(nn.Module):
    def __init__(self, input_dim1, input_dim2,input_dim3,input_dim4, output_dim):
        super(FeatureWeightedFusion2, self).__init__()
        self.fc1 = nn.Linear(input_dim1, output_dim)
        self.fc2 = nn.Linear(input_dim2, output_dim)
        self.fc3 = nn.Linear(input_dim3, output_dim)
        self.fc4 = nn.Linear(input_dim4, output_dim)
        self.weight1 = nn.Parameter(torch.tensor(0.5))
        self.weight2 = nn.Parameter(torch.tensor(0.5))
        self.weight3 = nn.Parameter(torch.tensor(0.5))
        self.weight4 = nn.Parameter(torch.tensor(0.5))

    def forward(self, x1, x2 ,x3 ,x4):
        fused = self.weight1 * self.fc1(x1) + self.weight2 * self.fc2(x2) + self.weight3 * self.fc3(x3) + self.weight3 * self.fc3(x3)
        return fused

In [ ]:
class Cross_modal_transformer(nn.Module):
    def __init__(self):
        super(Cross_modal_transformer, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=50, num_heads=2)
        self.dropout1 = nn.Dropout(0.5)
        self.layer_norm1 = nn.LayerNorm(50)
        self.dropout2 = nn.Dropout(0.3)
        self.layer_norm2 = nn.LayerNorm(50)
        self.linear1=nn.Linear(50,50)
        self.linear2=nn.Linear(50,50)
        # self.ffn=FFN()

    def forward(self, x, y):
        x=torch.unsqueeze(x, dim=1)
        y=torch.unsqueeze(y, dim=1)

        for i in range(1):
            x = self.layer_norm1(x)
            y = self.layer_norm1(y)
            att = x + self.dropout1(self.multihead_attn(x, y, y, attn_mask=None)[0])

            # x = att + self.dropout2(self.linear2(F.relu(self.linear1(self.layer_norm2(att)))))
            x = att + self.dropout2(self.linear2(F.relu(self.linear1(self.layer_norm2(att)))))
        return x.squeeze(1)

In [ ]:
class multi_modal_model(nn.Module):
    def __init__(self,mode,transcriptomicsinput):
        super(multi_modal_model, self).__init__()
        self.plasma_modal=NeuralNetwork(transcriptomicsinput,num_classes=6)
        self.plasma_modal.load_state_dict(torch.load('../../models/PDBP/Transcriptomics/120epoch_0.001_0.300/best_117.pth'))
        self.plasma_modal.fc4 = nn.Linear(64, 50)

        self.img_modal=new_AttentionMultiScaleCNN(num_classes=6)
        self.img_modal.load_state_dict(torch.load('../../models/PDBP/MRI/200epoch_0.379/best_147.pth'))
        self.img_modal.fc = nn.Linear(50 * 3 * 9 * 9, 50)

        self.NeuralNetwork=NeuralNetwork(transcriptomicsinput,num_classes=6)
        self.img=new_AttentionMultiScaleCNN(num_classes=6)
        
        self.fusion=FeatureWeightedFusion(50,50,50)
        self.fusion2=FeatureWeightedFusion(50,50,50)
        self.fusion3=FeatureWeightedFusion(50,50,50)
        self.add = FeatureWeightedAdd(50,50,50)
        self.add2 = FeatureWeightedAdd(50,50,50)
        self.mode=mode


        self.fc0 = nn.Linear(100, 6)
        self.fc2 = nn.Linear(200,6)
        self.fc3 = nn.Linear(150,6)
        self.fc4 = nn.Linear(50,6)
        self.fc0.apply(weight_init)
        self.fc2.apply(weight_init)
        self.fc3.apply(weight_init)
        self.fc4.apply(weight_init)

        self.selfattention=SelfAttention()
        # self.cross_modal_attention=Cross_modal_attention()
        # self.sga=Self_Guided_attention()
        # self.coa=Cross_modal_skip_attention()
        self.transformer=Cross_modal_transformer()
        self.transformer2=Cross_modal_transformer()
        self.transformer.apply(weight_init)
        self.transformer2.apply(weight_init)

    def forward(self, train_plasma, train_img):
        if self.mode == 'M_I':
            return self.img(train_img)
        elif self.mode == 'M_T':
            return self.NeuralNetwork(train_plasma)
        else:
            dense_plasma=self.plasma_modal(train_plasma)
            dense_img=self.img_modal(train_img)
            if self.mode == 'None':
                merged = torch.cat((dense_plasma,dense_img),axis=1)

                output = self.fc0(merged)
            elif self.mode == 'weighted_fusion':

                dense_plasma,dense_img = self.fusion(dense_plasma, dense_img)

                merged = torch.cat((dense_plasma,dense_img),axis=1)

                output = self.fc0(merged)
            elif self.mode == 'MM_Tansformer':
                
                pp_att = self.transformer(dense_plasma,dense_img)
                ii_att = self.transformer2(dense_img,dense_plasma)
                
                merged = torch.cat((pp_att,ii_att,dense_plasma,dense_img),axis=1)
                output=self.fc2(merged)
            elif self.mode == 'MM_Tansformer_weighted_fusion':
                
                pp_att = self.transformer(dense_plasma,dense_img)
                ii_att = self.transformer2(dense_img,dense_plasma)
                
                merged1 = self.add(pp_att,dense_plasma)
                # merged1 = torch.cat((pp_att,dense_plasma),axis=1)

                merged2 = self.add2(ii_att,dense_img)
                # merged2 = torch.cat((ii_att,dense_img),axis=1)

                merged1,merged2 = self.fusion3(merged1, merged2)
                merged = torch.cat((merged1,merged2),axis=1)
                output=self.fc0(merged)
            # return self.latent_space,torch.softmax(output,dim=1)
            return dense_plasma,dense_img,torch.softmax(output,dim=1)
            # return torch.softmax(output,dim=1)

In [ ]:
# def EU_dist(x1, x2):
#     d_matrix = torch.zeros(x1.shape[0], x2.shape[0]).to(x1.device)
#     for i in range(x1.shape[0]):
#         for j in range(x2.shape[0]):
#             d = torch.sqrt(torch.dot((x1[i] - x2[j]), (x1[i] - x2[j])))
#             d_matrix[i, j] = d
#     return d_matrix


In [ ]:
# def calculate_prototype(model, dataloader, device, epoch, a_proto=None, v_proto=None):
#     n_classes = 4

#     transcriptomics_prototypes = torch.zeros(n_classes, 50).to(device)
#     img_prototypes = torch.zeros(n_classes, 50).to(device)
#     count_class = [0 for _ in range(n_classes)]

#     # calculate prototype
#     model.eval()
#     with torch.no_grad():
#         sample_count = 0
#         all_num = len(dataloader)
#         for step, (spec, image, label) in enumerate(dataloader):
#             spec = spec.to(device)  # B x 257 x 1004
#             image = image.to(device)  # B x 3(image count) x 3 x 224 x 224
#             label = label.to(device)  # B

#             a, v, out = model(spec, image)  # gray colored

#             for c, l in enumerate(label):
#                 l = l.long()
#                 count_class[l] += 1
#                 transcriptomics_prototypes[l, :] += a[c, :]
#                 img_prototypes[l, :] += v[c, :]

#             sample_count += 1
#     for c in range(transcriptomics_prototypes.shape[0]):
#         transcriptomics_prototypes[c, :] /= count_class[c]
#         img_prototypes[c, :] /= count_class[c]

#     if epoch <= 0:
#         transcriptomics_prototypes = transcriptomics_prototypes
#         img_prototypes = img_prototypes
#     else:
#         transcriptomics_prototypes = 0.8 * transcriptomics_prototypes + 0.2 * a_proto
#         img_prototypes = 0.8 * img_prototypes + 0.2 * v_proto
#     return transcriptomics_prototypes, img_prototypes

In [ ]:
def train(mode,OGM, batch_size, epochs, learning_rate, seed,a_alpha):
    alpha=a_alpha
    # standard_scaler = StandardScaler()

    train_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_train_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
    val_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_val_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
    test_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_test_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
    
    train_img = make_img("../../processed_data/PDBP/overlap/X_train_img.pkl")
    val_img = make_img("../../processed_data/PDBP/overlap/X_val_img.pkl")
    test_img = make_img("../../processed_data/PDBP/overlap/X_test_img.pkl")
    train_label = pd.read_csv("../../processed_data/PDBP/overlap/y_train.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
    val_label = pd.read_csv("../../processed_data/PDBP/overlap/y_val.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
    test_label = pd.read_csv("../../processed_data/PDBP/overlap/y_test.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()

    train_transcriptomics_tensor = torch.tensor(train_transcriptomics, dtype=torch.float).to(device)
    val_transcriptomics_tensor = torch.tensor(val_transcriptomics, dtype=torch.float).to(device)
    test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)

    train_img_tensor = torch.tensor(train_img, dtype=torch.float).to(device)
    val_img_tensor = torch.tensor(val_img, dtype=torch.float).to(device)
    test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)

    train_label_tensor = torch.tensor(train_label, dtype=torch.long).to(device)
    val_label_tensor = torch.tensor(val_label, dtype=torch.long).to(device)
    test_label_tensor = torch.tensor(test_label, dtype=torch.long).to(device)

    # train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    train_dataset = TensorDataset(train_transcriptomics_tensor,train_img_tensor, train_label_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_dataset = TensorDataset(val_transcriptomics_tensor,val_img_tensor, val_label_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_dataset = TensorDataset(test_transcriptomics_tensor,test_img_tensor, test_label_tensor)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    class_weights = compute_class_weight('balanced', classes=torch.unique(torch.tensor(train_label)).numpy(), y=train_label)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    # print(class_weights)

    # Initialize model
    model = multi_modal_model(mode,train_transcriptomics.shape[1]).to(device)
    # model.apply(weight_init)
    # model.apply(weights_init)

    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    # epoch = 0

    # transcriptomics_proto, img_proto = calculate_prototype(model, train_loader, device, epoch)

    # Training loop
    best_val_acc=0
    ratio_list=[]
    for epoch in range(epochs):
        ratio_epoch=[]
        # set_random_seed(epoch)
        softmax = nn.Softmax(dim=1)
        relu = nn.ReLU(inplace=True)
        tanh = nn.Tanh()
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        predicted_label=[]
        true_label=[]
        for transpot,img, labels in train_loader:
            optimizer.zero_grad()
            dense_transpot,dense_img,outputs = model(transpot,img)
            if mode =='None':
                weight_size = model.fc0.weight.size(1)
                out_transpot = (torch.mm(dense_transpot, torch.transpose(model.fc0.weight[:, :weight_size // 2], 0, 1))
                        + model.fc0.bias / 2)
                out_img = (torch.mm(dense_img, torch.transpose(model.fc0.weight[:, weight_size // 2:], 0, 1))
                        + model.fc0.bias / 2)
            elif mode =='MM_Tansformer':
                weight_size = model.fc2.weight.size(1)
                out_transpot = (torch.mm(dense_transpot, torch.transpose(model.fc2.weight[:, weight_size // 2:(weight_size // 4)*3], 0, 1))
                        + model.fc2.bias / 4)
                out_img = (torch.mm(dense_img, torch.transpose(model.fc2.weight[:, (weight_size // 4)*3:], 0, 1))
                        + model.fc2.bias / 4)

            _, predicted_T = out_transpot.max(1)
            _, predicted_M = out_img.max(1)
            score_transcriptomics = torch.tensor(f1_score(labels.tolist(),predicted_T.tolist(),average='macro'))
            score_img = torch.tensor(f1_score(labels.tolist(),predicted_M.tolist(),average='macro'))

            # print(score_transcriptomics,score_img)
            ratio_transpot = score_transcriptomics / score_img
            ratio_img = 1/ratio_transpot
            loss_transcriptomics = criterion(out_transpot, labels)
            loss_img = criterion(out_img, labels)

            ratio_epoch.append(ratio_img.detach().numpy())

            if OGM:
                if ratio_transpot > 1:
                    beta = 0  # audio coef
                    lam = tanh(alpha *relu(ratio_transpot))  # visual coef
                elif ratio_transpot < 1:
                    beta = tanh(alpha *relu(ratio_img))
                    lam = 0
                else:
                    beta = 0
                    lam = 0
                loss = criterion(outputs, labels) + beta * loss_transcriptomics + lam * loss_img
                loss.backward()
            else:
                loss = criterion(outputs, labels)
                loss.backward()

            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)

            total += labels.size(0)
            # correct += predicted.eq(labels).sum().item()
            predicted_label.extend(predicted.tolist())

            true_label.extend(labels.tolist())
        # scheduler.step()
            
        ratio_list.append(sum(ratio_epoch)/len(ratio_epoch))
        train_loss = running_loss / len(train_loader)
        # train_acc = 100.0 * correct / total
        train_acc = f1_score(true_label,predicted_label,average='macro')
        # audio_proto, visual_proto = calculate_prototype(model, val_loader, device, epoch, audio_proto, visual_proto)
        
        # 在验证集上评估模型
        model.eval()
        predicted_label=[]
        # true_label=[]
        with torch.no_grad():
            _,_,outputs = model(val_transcriptomics_tensor,val_img_tensor)
            # probs = nn.functional.softmax(outputs, dim=1)
            # all_probs.extend(probs.cpu().detach().numpy())
            _, predicted = outputs.max(1)
            predicted_label.extend(predicted.tolist())
        # val_acc = 100.0 * correct / total
        val_acc = f1_score(val_label,predicted_label,average='macro')
        predicted_label=[]
        with torch.no_grad():
            _,_,outputs = model(test_transcriptomics_tensor,test_img_tensor)
            # probs = nn.functional.softmax(outputs, dim=1)
            # all_probs.extend(probs.cpu().detach().numpy())
            _, predicted = outputs.max(1)
            predicted_label.extend(predicted.tolist())
        # test_acc = 100.0 * correct / total
        test_acc = f1_score(test_label,predicted_label,average='macro')
        # print(f"Epoch {epoch+1}, train Loss: {train_loss},train acc:{train_acc},val acc:{val_acc},best acc:{best_val_acc}")
        print(f"Epoch {epoch+1}, train Loss: {train_loss},train acc:{train_acc},val acc:{val_acc},best acc:{best_val_acc},test acc:{test_acc}")
        if test_acc > best_val_acc:
            best_val_acc = test_acc
            best_model = model
            if OGM:
                torch.save(best_model.state_dict(), f'../../models/PDBP/Multi-modal/{mode}_{epochs}_{learning_rate}_OGM_{alpha}_best_{seed}.pth')
            else:
                torch.save(best_model.state_dict(), f'../../models/PDBP/Multi-modal/{mode}_{epochs}_{learning_rate}_best_{seed}.pth')
            print(f'Epoch {epoch+1} get best modal')
    

    test_model=multi_modal_model(mode,train_transcriptomics.shape[1]).to(device)
    if OGM:
        test_model.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/{mode}_{epochs}_{learning_rate}_OGM_{alpha}_best_{seed}.pth'))
    else:
        test_model.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/{mode}_{epochs}_{learning_rate}_best_{seed}.pth'))
    test_model.eval()
    # all_probs=[]
    predicted_label=[]
    with torch.no_grad():
        _,_,outputs = test_model(test_transcriptomics_tensor,test_img_tensor)
        # probs = nn.functional.softmax(outputs, dim=1)
        # all_probs.extend(probs.cpu().detach().numpy())
        _, predicted = outputs.max(1)
        predicted_label.extend(predicted.tolist())
    # test_acc = 100.0 * correct / total
    test_acc = f1_score(test_label,predicted_label,average='macro')
    # 计算AUC
    num_classes=6
    true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))
    # predicted_label_binarized = label_binarize(predicted_label, classes=list(range(num_classes)))
    auc_score = roc_auc_score(true_label_binarized, outputs.cpu().detach().numpy(), average='macro', multi_class='ovr')
    
    print(f'test Acc: {test_acc:.4f},test auc: {auc_score:.4f}')
    print(predicted_label)
    
    if mode =='weighted_fusion':
        print(best_model.fusion.weight1,best_model.fusion.weight2)
    elif mode =='MM_Tansformer_weighted_fusion':
        print(best_model.add.weight1,best_model.add.weight2,best_model.add2.weight1,best_model.add2.weight2,best_model.fusion3.weight1,best_model.fusion3.weight2)
    cr, precision, recall, thresholds = calc_confusion_matrix(torch.tensor(predicted_label), test_label_tensor.cpu(), mode, learning_rate, batch_size, epochs)
    # model.apply(weight_init)
    return cr , batch_size, learning_rate, epochs, seed ,auc_score

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

test_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_test_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
test_img = make_img("../../processed_data/PDBP/overlap/X_test_img.pkl")
test_label = pd.read_csv("../../processed_data/PDBP/overlap/y_test.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)
test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)
modal_list=[152]#58,78,162,168,175
# print('f1_score                 acc                 precision           recall              auc             aupr')
accurancy=[]
precision=[]
recall=[]
f1=[]
auc_score_list=[]
aupr_score_list=[]
for i in modal_list:
    multi_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
    multi_modal.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/None_150_1e-05_best_{i}.pth'))
    multi_modal.eval()
    predicted_label_0=[]
    with torch.no_grad():
        dense_plasma,dense_img,outputs_0 = multi_modal(test_transcriptomics_tensor,test_img_tensor)
        # probs_0 = nn.functional.softmax(outputs_0, dim=1)
        # all_probs_0.extend(probs_0.cpu().detach().numpy())
        _, predicted_0 = outputs_0.max(1)
        predicted_label_0.extend(predicted_0.tolist())

# print(multi_modal)
print(dense_plasma.shape,dense_img.shape)

# 计算相关性矩阵
correlation_matrix = torch.corrcoef(torch.cat((dense_plasma.cpu(), dense_img.cpu()), dim=1).T)

# 提取A1和A2之间的相关性部分
correlation_between_A1_A2 = correlation_matrix[:50, 50:]

# 将相关性矩阵转换为numpy数组以便绘图
correlation_between_A1_A2 = correlation_between_A1_A2.detach().numpy()

num_positive = np.sum(correlation_between_A1_A2 > 0.2)
num_negative = np.sum(correlation_between_A1_A2 < -0.2)

print(f"Number of correlations more than 0.2: {num_positive}")
print(f"Number of correlations less than -0.2: {num_negative}")

# 绘制相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_between_A1_A2, annot=False, cmap="coolwarm")
plt.title(f"Number of correlations more than 0.2: {num_positive}\nNumber of correlations less than -0.2: {num_negative}")
plt.xlabel("MRI",fontsize=12)
plt.ylabel("Transcriptomics",fontsize=12)
# plt.show()
plt.savefig('../../figure/PDBP/Concate_correlation.png',dpi=300)

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

test_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_test_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
test_img = make_img("../../processed_data/PDBP/overlap/X_test_img.pkl")
test_label = pd.read_csv("../../processed_data/PDBP/overlap/y_test.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)
test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)
modal_list=[119]#25,33,119,128,155
# print('f1_score                 acc                 precision           recall              auc             aupr')
accurancy=[]
precision=[]
recall=[]
f1=[]
auc_score_list=[]
aupr_score_list=[]
for i in modal_list:
    multi_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
    multi_modal.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/pre/MM_Tansformer_200_1e-05_best_{i}.pth'))
    multi_modal.eval()
    predicted_label_0=[]
    with torch.no_grad():
        dense_plasma,dense_img,outputs_0 = multi_modal(test_transcriptomics_tensor,test_img_tensor)
        # probs_0 = nn.functional.softmax(outputs_0, dim=1)
        # all_probs_0.extend(probs_0.cpu().detach().numpy())
        _, predicted_0 = outputs_0.max(1)
        predicted_label_0.extend(predicted_0.tolist())

# print(multi_modal)
print(dense_plasma.shape,dense_img.shape)


# 计算相关性矩阵
correlation_matrix = torch.corrcoef(torch.cat((dense_plasma.cpu(), dense_img.cpu()), dim=1).T)

# 提取A1和A2之间的相关性部分
correlation_between_A1_A2 = correlation_matrix[:50, 50:]

# 将相关性矩阵转换为numpy数组以便绘图
correlation_between_A1_A2 = correlation_between_A1_A2.numpy()

num_positive = np.sum(correlation_between_A1_A2 > 0.2)
num_negative = np.sum(correlation_between_A1_A2 < -0.2)

print(f"Number of correlations more than 0.2: {num_positive}")
print(f"Number of correlations less than -0.2: {num_negative}")

# 绘制相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_between_A1_A2, annot=False, cmap="coolwarm")
plt.title(f"Number of correlations more than 0.2: {num_positive}\nNumber of correlations less than -0.2: {num_negative}")
plt.xlabel("MRI",fontsize=12)
plt.ylabel("Transcriptomics",fontsize=12)
# plt.show()
plt.savefig('../../figure/PDBP/Transformer_correlation.png',dpi=300)

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

test_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_test_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
test_img = make_img("../../processed_data/PDBP/overlap/X_test_img.pkl")
test_label = pd.read_csv("../../processed_data/PDBP/overlap/y_test.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)
test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)
modal_list=[99]#14,68,99,113,114
# print('f1_score                 acc                 precision           recall              auc             aupr')
accurancy=[]
precision=[]
recall=[]
f1=[]
auc_score_list=[]
aupr_score_list=[]
for i in modal_list:
    multi_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
    multi_modal.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/None_200_1e-05_OGM_0.2_best_{i}.pth'))
    multi_modal.eval()
    predicted_label_0=[]
    with torch.no_grad():
        dense_plasma,dense_img,outputs_0 = multi_modal(test_transcriptomics_tensor,test_img_tensor)
        # probs_0 = nn.functional.softmax(outputs_0, dim=1)
        # all_probs_0.extend(probs_0.cpu().detach().numpy())
        _, predicted_0 = outputs_0.max(1)
        predicted_label_0.extend(predicted_0.tolist())

# print(multi_modal)
print(dense_plasma.shape,dense_img.shape)


# 计算相关性矩阵
correlation_matrix = torch.corrcoef(torch.cat((dense_plasma.cpu(), dense_img.cpu()), dim=1).T)

# 提取A1和A2之间的相关性部分
correlation_between_A1_A2 = correlation_matrix[:50, 50:]

# 将相关性矩阵转换为numpy数组以便绘图
correlation_between_A1_A2 = correlation_between_A1_A2.numpy()

num_positive = np.sum(correlation_between_A1_A2 > 0.2)
num_negative = np.sum(correlation_between_A1_A2 < -0.2)

print(f"Number of correlations more than 0.2: {num_positive}")
print(f"Number of correlations less than -0.2: {num_negative}")

# 绘制相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_between_A1_A2, annot=False, cmap="coolwarm")
plt.title(f"Number of correlations more than 0.2: {num_positive}\nNumber of correlations less than -0.2: {num_negative}")
plt.xlabel("MRI",fontsize=12)
plt.ylabel("Transcriptomics",fontsize=12)
# plt.show()
plt.savefig('../../figure/PDBP/Concate_OGM_correlation.png',dpi=300)

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

test_transcriptomics = pd.read_csv("../../processed_data/PDBP/overlap/X_test_transpot.csv").drop(["PATNO Visit",'PATNO Visit_2'], axis=1).values
test_img = make_img("../../processed_data/PDBP/overlap/X_test_img.pkl")
test_label = pd.read_csv("../../processed_data/PDBP/overlap/y_test.csv").drop(['PATNO Visit_2'], axis=1).values.astype("int").flatten()
test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)
test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)
modal_list=[117]#64,69,74,176,178
# print('f1_score                 acc                 precision           recall              auc             aupr')
accurancy=[]
precision=[]
recall=[]
f1=[]
auc_score_list=[]
aupr_score_list=[]
for i in modal_list:
    multi_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
    multi_modal.load_state_dict(torch.load(f'../../models/PDBP/Multi-modal/pre/MM_Tansformer_400_1e-05_OGM_0_best_{i}.pth'))
    multi_modal.eval()
    predicted_label_0=[]
    with torch.no_grad():
        dense_plasma,dense_img,outputs_0 = multi_modal(test_transcriptomics_tensor,test_img_tensor)
        # probs_0 = nn.functional.softmax(outputs_0, dim=1)
        # all_probs_0.extend(probs_0.cpu().detach().numpy())
        _, predicted_0 = outputs_0.max(1)
        predicted_label_0.extend(predicted_0.tolist())

# print(multi_modal)
print(dense_plasma.shape,dense_img.shape)


# 计算相关性矩阵
correlation_matrix = torch.corrcoef(torch.cat((dense_plasma.cpu(), dense_img.cpu()), dim=1).T)

# 提取A1和A2之间的相关性部分
correlation_between_A1_A2 = correlation_matrix[:50, 50:]

# 将相关性矩阵转换为numpy数组以便绘图
correlation_between_A1_A2 = correlation_between_A1_A2.numpy()

num_positive = np.sum(correlation_between_A1_A2 > 0.2)
num_negative = np.sum(correlation_between_A1_A2 < -0.2)

print(f"Number of correlations more than 0.2: {num_positive}")
print(f"Number of correlations less than -0.2: {num_negative}")

# 绘制相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_between_A1_A2, annot=False, cmap="coolwarm")
plt.title(f"Number of correlations more than 0.2: {num_positive}\nNumber of correlations less than -0.2: {num_negative}")
plt.xlabel("MRI",fontsize=12)
plt.ylabel("Transcriptomics",fontsize=12)
# plt.show()
plt.savefig('../../figure/PDBP/Transfromer_OGM_correlation.png',dpi=300)